
# 형태소 분석 기반 토큰화의 문제

- 형태소 분석 기반 토큰화는 **"이미 알고 있는 단어"** 를 중심으로 판단한다. 그래서 오탈자나 띄어쓰기 실수, 신조어, 외래어, 고유어에 대해서 취약한 단점이 있다.
- 이로 인해 발생 할 수 있는 잠재적 문제점은:
    - 어휘사전이 불필요하게 커진다.
        - 같은 의미의 단어가 형태소 분석이 안되어 여러개 등록될 수있다.
        - ex) 신조어 `돈쭐` 이라는 단어를 인식 못할 경우 `"돈쭐내러", "돈쭐나", "돈쭐냄"` 등이 다 등록 될 수 있다.
    - OOV(Out Of Vocab)에 대응하기 어렵게 만든다.
        - 같은 어근의 단어가 있지만 조사등이 바뀐 신조어등을 OOV로 인식할 수있다.


## 어휘 사전(Vocabulary)과 Out Of Vocabulary (OOV)

- 어휘사전(Vocab)은 토크나이저(Tokenizer)가 사용하는 모든 토큰의 집합이며,**각 토큰**을 고유한 **정수 ID**에 매핑한 사전이다. 토크나이저가 텍스트를 토큰 ID 시퀀스로 변환할 때 기준으로 사용된다. 

   - 매핑된 정수는 모델에 입력되는 텍스트 데이터를 숫자 형식으로 변환해 모델이 처리할 수 있도록 돕는다.
   - 예
        ```bash
        {
            "자연어": 0,
            "처리": 1,
            "는": 2,
            "재미있다": 3,
            "공부": 4,
        }
        ```
- **Out Of Vocabulary (OOV)**
   - 어휘 사전(Vocab): 코퍼스를 구성하는 모든 토큰의 집합.
   - **OOV**란 어휘 사전에 포함되지 않은 토큰을 의미하며, 모델이 해당 토큰을 처리할 수 없기 때문에 일반적으로 특별한 토큰(예: `[UNK]`)으로 대체되거나 다른 방식으로 처리된다.

# Subword Tokenization(하위 단어 토큰화)

## 정의

- Subword Tokenization은 단어를 더 작은 단위(subword)로 나누어 텍스트를 토큰화하는 방식이다.  
    - subword는 하나의 단어를 구성하는 단어들을 말한다.(coworker: co, work, er)
- 주로 자주 등장하는 단어의 일부를 공통된 토큰으로 만들고, 희귀하거나 복합적인 단어는 작은 조각(subword)으로 나누어 처리한다.
- 단어 자체를 그대로 사용하기보다는 단어의 일부를 나누어 처리함으로써 새로운 단어나 미등록 단어(Out-of-Vocabulary) 문제를 줄일 수 있다.

## 장점

1. **미등록 단어 처리 가능**  
   -  새로운 단어(신조어, 속어, 고유어등)가 등장해도 미리 정의된 subword를 조합해서 표현할 수 있어 OOV 문제를 줄일 수 있다.  

2. **어휘 크기 축소**  
   - 같은 subword를 여러 단어에서 공유함으로써, 완전한 단어를 사용하는 경우보다 어휘집의 크기를 작게 유지할 수 있다.


## 종류

1. **Byte-Pair Encoding (BPE)**  
   - 자주 등장하는 문자 쌍을 반복적으로 병합해 서브워드를 생성하는 방식.
   - OpenAI의 GPT 모델에 사용된 토크나이저이다.

2. **Unigram**  
   - 빈도기반 확률모델에 따라 subword 단위를 선택하는 방식이다.  
   - BPE보다 유연하여 더 다양한 분할 결과를 얻을 수 있다.

3. **WordPiece**  
   - BPE와 유사하지만, 빈도수가 아니라, 가능성이 높은 조합(합쳐질 가능성이 높은 subword)에 기반해 subword들을 찾는다.
   - Google의 BERT 모델에 사용된 토크나이저이다.

# Byte Pair Encoding 방식

- 원래 Text data 압축을 위해 만들어진 방법으로 text 에서 자주 같이 나오는 두 글짜 쌍을 합쳐서 하나의 부호(기호,글자)로 만들어 나가면서 글자 수를 줄이는 알고리즘이다. 
- 연속된 글자 쌍이 더 나타나지 않거나 정해진 어휘사전 크기에 도달 할 때 까지 조합을 찾아 부호화 하는 작업을 반복한다.

## text 압축 방식의 예
- 원문: abracadabra
1. AracadAra: ab -> A :=> 원문에서 가장 빈도수 많은 ab를 A(부호로 아무 글자나 사용할 수 있다.)로 치환
2. ABcadAB: ra -> B :=> 1에서 가장 빈도수가 많은 ra를 B로 치환
3. CcadC: AB -> C :=> 2에서 가장 빈도수 맣은 AB를 C로 치환한다.(치환된 글자 쌍도 변환대상에 포함된다.)

## BPE Tokenizer 방식
BPE 토크나이저는 자주 등장하는 글자 쌍을 찾아 치환하는 대신 **단어 사전**에 추가한다.

### 예)
1. 1차적으로 토큰화를 진행한다. 일반적으로 공백 기준으로 토큰화를 한다.
    - BPE 방식은 1차적으로 진행한 토큰 내에서 글자쌍을 만들어 어휘사전에 등록한다.
    - 1차 토큰화 결과가 다음과 같다고 가정하자.
      - 각 토큰과 빈도수
        - ('low', 5), ('lower', 2), ('newest', 6), ('widest', 3) 
      - 'low', 'lower', 'newest', 'widest' 로 구성됨.
2. 모든 토큰들을 글자 단위로 나누고 어휘사전에 추가한다.
    - ('l', 'o', 'w',  5), ('l', 'o', 'w', 'e', 'r', 2), ('n', 'e', 'w', 'e', 's', 't', 6), ('w', 'i', 'd', 'e', 's', 't', 3)
    - 어휘사전: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']
3. 토큰 별로 가장 자주 등장하는 글자 쌍(byte pair)를 찾는다.  위에서는 **'es'가 총 9번으로 가장 많이 등장함**. 'e'와 's'를 'es'로 합치고 어휘 사전에 추가한다.
    - ('l', 'o', 'w',  5), ('l', 'o', 'w', 'e', 'r', 2), ('n', 'e', 'w', **'es'**, 't', 6), ('w', 'i', 'd', **'es'**, 't', 3)
    - 어휘사전: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w', **'es'**]
4. 3 번의 과정을 계속 반복한다. 빈도수가 가장 많은 'es'와 't' 쌍을 'est'로 병합하고 'est'를 어휘 사전에 추가한다.
    - ('l', 'o', 'w',  5), ('l', 'o', 'w', 'e', 'r', 2), ('n', 'e', 'w', **'est'**, 6), ('w', 'i', 'd', **'est'**, 3)
    - 어휘사전: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w', **'es'**, **'est'**]
5. 만약 10번 반복했다고 하면 다음과 같은 빈도 사전과 어휘 사전이 생성된다.
    - 빈도 사전: (**'low'**, 5), (**'low'**, 'e', 'r', 2), ('n', 'e', 'w', **'est'**, 6), ('w', 'i', 'd', **'est'**, 3)
    - 어휘사전: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w', **'es'**, **'est'**, **'lo'**,**'low'**, **'ne'**, **'new'**, **'newest'**, **'wi'**, **'wid'**, **'widest'**]

- 위와 같이 어휘 사전이 만들어 지면 원래 어휘서전에 없던 것들에 대한 처리를 할 수있다.
    - ex)
        - 'newer' :=> 'new', 'e', 'r', 
        - 'lowest' :=> 'low', 'est'
        - 'wider' :=> 'wid', 'e', 'r'

In [ ]:
100/(1000*2000) 0.00005

5e-05

In [2]:
5/(5*5)

0.2

# WordPiece tokenizer

- Byte Pair Encoding 이 빈도 기반이라면 wordpiece tokenizer는 확률 기반으로 글자 쌍을 병합한다.
- 두개 글자 쌍의 빈도수를 각 개별 글자 빈도수의 곱으로 나눈 점수가 가장 높은 순서대로 글자쌍을 묶어 나간다.

$$
score = \cfrac{f(x, y)}{f(x)\cdot f(y)} 
$$

함수 f는 빈도를 나타내며 x, y는 병합하려는 하위 단어이다.

- 각 토큰을 글자 단위로 분해 후 어휘사전에 추가
  -  ('l','o','w', 5), ('l','o','w', 'e', 'r', 2), ('n', 'e', 'w', 'e', 's', 't', 6), ('w', 'i', 'd', 'e', 's', 't', 3)
  - 어휘사전: ('d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w')
- 가장 빈도수가 높은 쌍은 'e','s'로 9번 등장한다. 이때 각 글자는 전체에서 각각 'e'는 17번, 's'는 9번 등장한다. 위 공식에 대입하면 score는 $\frac{9}{17 \times 9} \approx 0.06$ 이다.
- 'i'와 'd' 쌍은 3번만 등장하지만 전체에서 각각 'i' 3번, 'd' 3번 등장한다. 그래서 score는 $\frac{3}{3 \times 3} \approx 0.33$ 이다.
- 나타난 빈도수는 'es' 가 많치만 더 높은 score를 가지는 'id' 쌍을 병합한다.
  -  ('l','o','w', 5), ('l','o','w', 'e', 'r', 2), ('n', 'e', 'w', 'e', 's', 't', 6), ('w', **'id'**, 'e', 's', 't', 3)
  - 어휘사전: ('d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w', **'id'**)
위의 작업을 반복해 연속된 글자 쌍이 더이상 나타나지 않거나 어휘 사전 max 크기에 도달할 때 까지 학습한다.

# Unigram 방식
- 빈도 기반 확률 모델을 사용하여 효율적으로 서브워드를 선택하고, 불필요한 서브워드를 제거해 최적의 어휘 크기를 찾는 알고리즘

- **초기 어휘 집합 구성**
    - 대상 text에 모든 단어와 그 서브스트링을 포함한 어휘 집합을 생성한다. 이 어휘 집합은 나올 수있는 모든 subword들을 다 모아놓은 것이다. 
    - 예를 들어 "hug" 단어의  ["h", "u", "g", "hu", "ug", "hug"]  substring을 만든다. 이들이 subword 후보가 된다.
- **각 Subword의 빈도수 기반 확률 계산**
    -  $\cfrac{subword가\;나타난\;횟수}{전체\;빈도수}$ 로 각 subword들의 나타난 확률을 계산한다.
- **가능한 분할에 대한 확률 계산**
    - 단어를 여러 서브워드로 분할할 수 있는 경우, 각 분할에 대한 전체 확률을 계산한다.
    - 확률 계산은 $ P(subword1)\;\times \; P(subword2)\;\times\; ..$ 으로 계산한다.
    - 예를 들어 "hug" 를 분할 한다고 했을 때
        1. \["h", "u", "g"\]: $ P(h) \times P(u) \times P(g) $
        2. \["hu", "g"\]: $ P(hu) \times P(g) $

   - 각각의 확률을 계산한 후, **가장 높은 확률**을 가진 분할을 선택한다.
     - 위 예에서 만약 1의 확률이 0.01 이고 2의 확률이 0.00001 이라면 첫번째 분할이 선택된다.

- **서브워드 제거**
    - 위의 훈련과정에서 불필요한 서브워드를 제거하면서 최적의 어휘 집합을 찾아간다. 
    - 제거 대상은 빈도수가 낮거나 조합에 크게 영향을 주지 않은 subword들이다.

> ### korpora 말뭉치
> - 다양한 한글 데이터셋을 제공하는 패키지
> - `pip install korpora`

In [4]:
!uv pip install korpora

Resolved 11 packages in 267ms
Prepared 7 packages in 258ms
Installed 8 packages in 347ms
 + certifi==2026.5.20
 + charset-normalizer==3.4.7
 + dataclasses==0.8
 + idna==3.18
 + korpora==0.2.0
 + requests==2.34.2
 + urllib3==2.7.0
 + xlrd==2.0.2


In [5]:
from Korpora import Korpora

# 청와대 국민 청원 데이터셋
corpus = Korpora.load("korean_petitions")
corpus


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : Hyunjoong Kim lovit@github
    Repository : https://github.com/lovit/petitions_archive
    References :

    청와대 국민청원 게시판의 데이터를 월별로 수집한 것입니다.
    청원은 게시판에 글을 올린 뒤, 한달 간 청원이 진행됩니다.
    수집되는 데이터는 청원종료가 된 이후의 데이터이며, 청원 내 댓글은 수집되지 않습니다.
    단 청원의 동의 개수는 수집됩니다.
    자세한 내용은 위의 repository를 참고하세요.

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[korean_petitions] download petitions_2017-08: 1.84MB [00:00, 22.4MB/s]
[korean_petitions] download petitions_2017-09: 20.4MB [00:00, 42.5MB/s]                            
[korean_petitions] download petitions_2017-10: 12.0MB [00:00, 55.5MB/s]                            
[korean_petitions] download petitions_2017-11: 28.4MB [00:00, 53.7MB/s]                            
[korean_petitions] download petitions_2017-12: 29.0MB [00:00, 65.5MB/s]                            
[korean_petitions] download petitions_2018-01: 43.9MB [00:00, 66.7MB/s]                            
[korean_petitions] download petitions_2018-02: 33.8MB [00:00, 38.6MB/s]                            
[korean_petitions] download petitions_2018-03: 34.3MB [00:01, 18.2MB/s]                            
[korean_petitions] download petitions_2018-04: 35.5MB [00:01, 19.4MB/s]                            
[korean_petitions] download petitions_2018-05: 37.5MB [00:03, 10.9MB/s]                            
[korean_petitions] download 

In [6]:
petitions = corpus.get_all_texts()
type(petitions), len(petitions)

(list, 433631)

In [10]:
print(petitions[10010])

안전한 국가. 피해자가 정말로 잊어내고 새출발 할수 있게..도와주는 나라..두려움에 떨지않게 신속하게 대처해주시고 제발..부탁드립니다..


In [11]:
import os

os.makedirs("data/petitions", exist_ok=True)
with open("data/petitions/petitions_corpus.txt", "wt", encoding="utf-8") as fw:
    for p in petitions:
        fw.write(p+"\n")

In [1]:
with open("data/petitions/petitions_corpus.txt", "rt", encoding="utf-8") as fr:
    load_petitions = fr.read()

In [2]:
print(len(load_petitions))
load_petitions[:100]


222421175


"안녕하세요. 현재 사대, 교대 등 교원양성학교들의 예비교사들이 임용절벽에 매우 힘들어 하고 있는 줄로 압니다. 정부 부처에서는 영양사의 영양'교사'화, 폭발적인 영양'교사' 채용,"

# Hugging Face tokenizers 패키지 사용해 토큰화 처리

- Hugging face는 
  - Subword방식의 토크나이저를 생성할 수 있는 알고리즘을 제공한다. `tokenizers` library를 이용해 사용할 수 있다.
  - GPT, BERT등 다양한 LLM 모델에서 사용된 Pretrained Tokenizer들을 제공한다. `transformers` Library를 이용해 사용할 수 있다.
- 설치: `pip install tokenizers`
- Documentation: https://huggingface.co/docs/tokenizers/index
- Tokenizer 생성
    - 토큰화 알고리즘을 지정해 instance 생성.
- Trainer 생성
    - 학습 파라미터를 설정해서 instance 생성
- Tokenizer 학습
    - train() 메소드: 학습 text 파일 경로를 지정해서 학습
    - train_from_iterator() 메소드: 학습할 string들을 iterator를 통해 제공.
- https://github.com/huggingface/tokenizers

In [3]:
!uv pip install tokenizers

Resolved 24 packages in 364ms
 Downloaded hf-xet
 Downloaded tokenizers
Prepared 12 packages in 496ms
Installed 18 packages in 956ms
 + annotated-doc==0.0.4
 + anyio==4.13.0
 + click==8.4.1
 + filelock==3.29.1
 + fsspec==2026.4.0
 + h11==0.16.0
 + hf-xet==1.5.1
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.18.0
 + markdown-it-py==4.2.0
 + mdurl==0.1.2
 + pyyaml==6.0.3
 + rich==15.0.0
 + shellingham==1.5.4
 + tokenizers==0.23.1
 + typer==0.25.1
 + typing-extensions==4.15.0


In [23]:
import time

from tokenizers import Tokenizer # 문서를 토큰화(어휘사전을 바탕으로) 하는 객체
from tokenizers.models import BPE # 토큰화 subword 알고리즘.
# Subword방식으로 토큰화 하기 전에 문서를 미리 토큰화할때 사용하는 방식.
from tokenizers.pre_tokenizers import Whitespace 
from tokenizers.trainers import BpeTrainer 
# Tokenizer 학습시키는 Trainer객체(토크나이저모델 학습기.)

In [24]:
# 토크나이저를 생성 - 모델을 넣어서 생성.
tokenizer = Tokenizer(
    BPE(unk_token='[UNK]') # unk 토큰등록해서 생성. unk-unkown 토큰: 어휘사전에 없는 토큰(단어)
)
# Pre-Tokenizer를 추가
tokenizer.pre_tokenizer = Whitespace()

In [25]:
# Trainer 생성
# Trainer역할 - 토크나이저 학습 => 어휘사전을 생성.
trainer = BpeTrainer(
    vocab_size=20_000, # 어휘사전
    min_frequency=10,  # 사전에 추가할 때 최소 출연 횟수(빈도수).
    special_tokens=["[UNK]", "[PAD]"], # 어휘사전에 추가할 특수목적 토큰들 설정. 
                                       # unk_token에 설정한 것은 반드시 추가한다.
    
    # 연결 토큰과 어절시작토큰을 분리해서 저장. 연결토큰앞에 붙일 접두어 설정                
    continuing_subword_prefix="##", 
)
# special token (특수목적 토큰)
# - 토크나이저나 언어처리모델이 특정 목적으로 사용할 토큰을 말한다.
# - 학습을 통해 어휘사전에 추가되는 것이 아니라 사람이 직접 추가한다.
# - 보통 special token은 [] 나 <> 로 묶어준다.
# - 대표적인 special token
#   - [UNK], <unk>: OOV토큰 표시 (어휘사전에 없는 토큰)
#   - [PAD]: Padding 을 표시하는 토큰. 문서의 토큰개수를 맞추기 위해서 개수가 모자랄때 채우는 토큰.
#   - [SOS]: 문장의 시작을 표시하는 토큰.
#   - [EOS]: 문장의 종료를 표시하는 토큰
#   - <cls>: 문서의 시작과 문서의 의미를 표현하는 토큰 (BERT 모델이 사용하는 특수토큰)
#   - <mask>: 문서의 일부 내용들을 가리는 용도의 특수토큰.

In [26]:
# Trainer를 이용해서 Tokenizer를 학습 (어휘사전 생성) - 학습시킬 데이터셋(말뭉치) 전달
s = time.time()
# 학습시킬 문서 파일들의 경로를 리스트로 묶어서 전달
tokenizer.train(
    ["data/petitions/petitions_corpus.txt"], 
    trainer=trainer
)

e = time.time()
print(f"걸린시간: {e-s}초")

걸린시간: 33.54149603843689초


In [39]:
import os
# 토크나이저를 파일로 저장: 어휘사전 + 토크나이저관련설정 (json 형식)
os.makedirs('saved_model/tokenizers', exist_ok=True)
save_path = "saved_model/tokenizers/petitions_bpe.json"

tokenizer.save(save_path)

In [27]:
# 저장된 토크나이저를 load해서 Tokenizer객체 생성.
from tokenizers import Tokenizer
load_tokenzier = Tokenizer.from_file(save_path)

In [29]:
# 총 어휘수
tokenizer.get_vocab_size()

20000

In [16]:
# 어휘사전 조회
# tokenizer.get_vocab()

In [30]:
# 테스트 문장
sports_txt = "프리미어리그 역대 개인 최다골 기록을 보유하고 있는 시어러가 손흥민의 골 결정력을 재차 극찬했다."
petition_txt = "이 글을 쓴 이유는 다름아닌 '전안법'시행 반대를 주장하기 위해서입니다. 먼저, '전안법'은 전기용품 및 생활용품을 판매하는 업체에서 KC인증마크를 의무적으로 받는 것입니다."
comment_txt = "멋진 식사를 즐기기에 좋은 장소 - 채식 메뉴가 정말 훌륭했습니다. 당근 케이크는 아마도 내가 먹어본 디저트 중 최고였을 거예요."

In [34]:
# 토큰화: 문서(string) -> 토큰들로 나누기. 
token_output = tokenizer.encode(sports_txt)
token_output # Encoding 타입객체

Encoding(num_tokens=34, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [35]:
sports_txt

'프리미어리그 역대 개인 최다골 기록을 보유하고 있는 시어러가 손흥민의 골 결정력을 재차 극찬했다.'

In [36]:
# 토큰화 결과를 확인
tokens = token_output.tokens
tokens

['프',
 '##리',
 '##미',
 '##어',
 '##리',
 '##그',
 '역',
 '##대',
 '개인',
 '최',
 '##다',
 '##골',
 '기록',
 '##을',
 '보유',
 '##하고',
 '있는',
 '시',
 '##어',
 '##러',
 '##가',
 '손',
 '##흥',
 '##민',
 '##의',
 '골',
 '결정',
 '##력을',
 '재',
 '##차',
 '극',
 '##찬',
 '##했다',
 '.']

In [37]:
# 토큰 id(정수) 로 조회
token_ids = token_output.ids
print(token_ids)

[7390, 8336, 8387, 8204, 8336, 8518, 6180, 8275, 15087, 6891, 8259, 8781, 16821, 8243, 16333, 14892, 14904, 5908, 8204, 8367, 8226, 5802, 9006, 8288, 8223, 4027, 15622, 15268, 6449, 8295, 4129, 8576, 15372, 15]


In [22]:
petition_txt

"이 글을 쓴 이유는 다름아닌 '전안법'시행 반대를 주장하기 위해서입니다. 먼저, '전안법'은 전기용품 및 생활용품을 판매하는 업체에서 KC인증마크를 의무적으로 받는 것입니다."

In [23]:
token_output2 = tokenizer.encode(petition_txt)
print(token_output2)
token_output2.tokens

Encoding(num_tokens=52, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])


['이',
 '글을',
 '쓴',
 '이유는',
 '다',
 '름',
 '아닌',
 "'",
 '전',
 '안',
 '법',
 "'",
 '시행',
 '반대',
 '를',
 '주장',
 '하기',
 '위해서',
 '입니다',
 '.',
 '먼저',
 ',',
 "'",
 '전',
 '안',
 '법',
 "'",
 '은',
 '전기',
 '용',
 '품',
 '및',
 '생활',
 '용',
 '품',
 '을',
 '판매',
 '하는',
 '업체',
 '에서',
 'K',
 'C',
 '인',
 '증',
 '마',
 '크',
 '를',
 '의무',
 '적으로',
 '받는',
 '것입니다',
 '.']

In [24]:
token_output2.ids

[6396,
 8577,
 6044,
 9246,
 4563,
 5107,
 8323,
 8,
 6476,
 6073,
 5412,
 8,
 8656,
 8658,
 5101,
 8953,
 8301,
 8580,
 8212,
 15,
 8855,
 13,
 8,
 6476,
 6073,
 5412,
 8,
 6360,
 9353,
 6278,
 7375,
 5345,
 8437,
 6278,
 7375,
 6364,
 9118,
 8208,
 8692,
 8210,
 44,
 36,
 6400,
 6625,
 5140,
 7091,
 5101,
 8521,
 8253,
 8546,
 8299,
 15]

In [25]:
# 개별 토큰 문자열의 ID를 조회
tokenizer.token_to_id("이유는")

9246

In [26]:
# ID를 문자열로 조회
tokenizer.id_to_token(9246)

'이유는'

In [ ]:
# token id 리스트 -> 문자열(문서 str)
tokenizer.decode(token_ids)

'프 리 미 어 리 그 역 대 개인 최 다 골 기록 을 보유 하고 있는 시 어 러 가 손 흥 민 의 골 결정 력을 재 차 극 찬 했다 .'

In [31]:
' '.join(tokens)

'프 리 미 어 리 그 역 대 개인 최 다 골 기록 을 보유 하고 있는 시 어 러 가 손 흥 민 의 골 결정 력을 재 차 극 찬 했다 .'

In [33]:
sports_txt

'프리미어리그 역대 개인 최다골 기록을 보유하고 있는 시어러가 손흥민의 골 결정력을 재차 극찬했다.'

In [1]:
##################################
# Wordpiece 방식으로 토크나이저 학습
##################################
from tokenizers.models import WordPiece # , BPE
from tokenizers.trainers import WordPieceTrainer #, BpeTrainer
from tokenizers import Tokenizer
from tokenizers.pre_tokenizers import Whitespace
import time

In [2]:
# Tokenizer생성 - 모델(WordPiece)을 넣어서 생성.
w_tokenizer = Tokenizer(
    WordPiece(unk_token="<unk>")
)
# Pre tokenizer 설정
w_tokenizer.pre_tokenizer = Whitespace()

In [ ]:
# trainer 설정
w_trainer = WordPieceTrainer(
    vocab_size=20_000,
    special_tokens=["<unk>", "<pad>", "<sos>", "<eos>", "<mask>"]
)

In [4]:
# 학습
s = time.time()

w_tokenizer.train(
    ["data/petitions/petitions_corpus.txt"],   #, "a.txt", "b.txt"],
    trainer=w_trainer
)

print(f"걸린시간: {time.time() - s}초")

걸린시간: 22.753894329071045초


In [5]:
# 저장
save_path = "saved_model/tokenizers/petitions_wordpiece.json"
w_tokenizer.save(save_path)

In [6]:
# 로드
load_w_tokenizer = Tokenizer.from_file(save_path)

In [11]:
w_tokenizer.get_vocab_size()
# w_tokenizer.get_vocab()

20000

In [12]:
# token(str) -> id 조회
w_tokenizer.token_to_id("##수의")

15988

In [13]:
# id -> token
w_tokenizer.id_to_token(12316)

'##곸'

In [16]:
# 토큰화: 인코딩 (문서->토큰)
encoding = w_tokenizer.encode(sports_txt)
encoding

Encoding(num_tokens=34, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [17]:
# token들 조회
encoding.tokens

['프',
 '##리',
 '##미',
 '##어',
 '##리',
 '##그',
 '역',
 '##대',
 '개인',
 '최',
 '##다',
 '##골',
 '기록',
 '##을',
 '보유',
 '##하고',
 '있는',
 '시',
 '##어',
 '##러',
 '##가',
 '손',
 '##흥',
 '##민',
 '##의',
 '골',
 '결정',
 '##력을',
 '재',
 '##차',
 '극',
 '##찬',
 '##했다',
 '.']

In [18]:
encoding.ids

[7393,
 8280,
 8705,
 8207,
 8280,
 8269,
 6183,
 8253,
 15090,
 6894,
 8215,
 8950,
 16824,
 8211,
 16336,
 14895,
 14907,
 5911,
 8207,
 8301,
 8246,
 5805,
 8510,
 8642,
 8222,
 4030,
 15625,
 15271,
 6452,
 8277,
 4132,
 9007,
 15375,
 18]

In [19]:
# encoding ids 목록으로 다시 문자열로 변환: 디코딩
result = w_tokenizer.decode(encoding.ids)
result


'프 ##리 ##미 ##어 ##리 ##그 역 ##대 개인 최 ##다 ##골 기록 ##을 보유 ##하고 있는 시 ##어 ##러 ##가 손 ##흥 ##민 ##의 골 결정 ##력을 재 ##차 극 ##찬 ##했다 .'

In [22]:
result2 = ""
for token_id in encoding.ids:
     token = w_tokenizer.id_to_token(token_id)
     if token.startswith("##"): # 연결토큰
          result2 += token[2:]
     else: # 어절 시작토큰
          result2 += " "+token

print(result2.strip())

프리미어리그 역대 개인 최다골 기록을 보유하고 있는 시어러가 손흥민의 골 결정력을 재차 극찬했다 .


In [ ]:
###################################
# Unigram 토크나이저 학습
###################################

In [9]:
sports_txt = "프리미어리그 역대 개인 최다골 기록을 보유하고 있는 시어러가 손흥민의 골 결정력을 재차 극찬했다."
petition_txt = "이 글을 쓴 이유는 다름아닌 '전안법'시행 반대를 주장하기 위해서입니다. 먼저, '전안법'은 전기용품 및 생활용품을 판매하는 업체에서 KC인증마크를 의무적으로 받는 것입니다."
comment_txt = "멋진 식사를 즐기기에 좋은 장소 - 채식 메뉴가 정말 훌륭했습니다. 당근 케이크는 아마도 내가 먹어본 디저트 중 최고였을 거예요."

In [3]:
from tokenizers import Tokenizer
from tokenizers.models import Unigram   #, BPE, WordPiece
from tokenizers.trainers import UnigramTrainer  #, BpeTrainer, WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace
import time

In [5]:
# 토크나이저 객체 생성
## Unigram()은 학습을 위해서 생성할 경우 unk_token을 설정하지 않는다.
### Trainer의 Special Token중에 0번 토큰을 unk_token으로 사용한다.
u_tokenizer = Tokenizer(
    Unigram()
)
u_tokenizer.pre_tokenizer = Whitespace()

# Train 생성
u_trainer = UnigramTrainer(
    vocab_size=20000,
    special_tokens=["<unk>", "<pad>"], 
    min_frequency=10
)

In [6]:
# 학습
s = time.time()

u_tokenizer.train(
    ["data/petitions/petitions_corpus.txt"],
    trainer=u_trainer
)

print(f"걸린시간: {time.time() - s}초")

걸린시간: 215.0078935623169초


In [7]:
# 저장
save_path = "saved_model/tokenizers/petitions_unigram.json"
u_tokenizer.save(save_path)

In [8]:
# 로드
load_u_tokenizer = Tokenizer.from_file(save_path)

In [10]:
comment_txt

'멋진 식사를 즐기기에 좋은 장소 - 채식 메뉴가 정말 훌륭했습니다. 당근 케이크는 아마도 내가 먹어본 디저트 중 최고였을 거예요.'

In [11]:
# 인코딩
encoding = u_tokenizer.encode(comment_txt)
encoding.tokens

['멋진',
 '식사',
 '를',
 '즐기',
 '기에',
 '좋은',
 '장소',
 '-',
 '채',
 '식',
 '메뉴',
 '가',
 '정말',
 '훌륭',
 '했습니다',
 '.',
 '당',
 '근',
 '케이',
 '크',
 '는',
 '아마도',
 '내가',
 '먹어',
 '본',
 '디',
 '저',
 '트',
 '중',
 '최고',
 '였',
 '을',
 '거예요',
 '.']

In [12]:
encoding.ids

[11930,
 4704,
 11,
 5685,
 1047,
 317,
 2275,
 66,
 507,
 249,
 12447,
 9,
 131,
 12592,
 171,
 2,
 158,
 839,
 6648,
 1087,
 10,
 6334,
 651,
 5650,
 274,
 1175,
 134,
 421,
 56,
 1712,
 2124,
 5,
 10838,
 2]